# 🔬 NammaRetina - Diabetic Retinopathy Training Pipeline
This notebook mounts Google Drive, extracts the preprocessed dataset, sets up EfficientNetB0, and runs our 3-Stage Training Pipeline to classify diabetic retinopathy (5 classes, stages 0-4).

### ⚙️ Hardware Check
Before running, make sure you have GPU acceleration enabled:
1. Go to **Runtime** > **Change runtime type**
2. Choose **T4 GPU** under Hardware accelerator
3. Click **Save**

In [ ]:
# Check if GPU is active
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPUs Available:", gpus)
if not gpus:
    print("⚠️ WARNING: GPU not active. Please enable GPU in Runtime settings!")

## 📂 Step 1: Mount Drive and Extract Preprocessed Data
We mount Google Drive and unzip `processed.zip` to local storage on Colab `/content` (which uses fast SSDs).

In [ ]:
from google.colab import drive
import os
import shutil

# Mount Drive
drive.mount('/content/drive')

# Copy and unzip the preprocessed zip
zip_path = '/content/drive/MyDrive/NammaRetina/processed.zip'
extract_path = '/content/dataset'

if not os.path.exists(zip_path):
    print(f"❌ ERROR: {zip_path} not found in your Google Drive.")
    print("Please upload processed.zip to the 'NammaRetina' folder in Google Drive first!")
else:
    print("📦 Copying zip from Drive...")
    os.makedirs(extract_path, exist_ok=True)
    shutil.copy(zip_path, '/content/processed.zip')
    
    print("🔓 Unzipping dataset...")
    !unzip -q /content/processed.zip -d /content/dataset/
    
    # Clean up zip from /content/
    os.remove('/content/processed.zip')
    print("✅ Done! Dataset extracted to /content/dataset")

## 🧪 Step 2: Import Libraries & Configure Paths

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback

# --- Paths aligned with processed/ folder structure ---
DATA_DIR = '/content/dataset/processed'
TRAIN_DF_PATH = os.path.join(DATA_DIR, 'train.csv')
VAL_DF_PATH = os.path.join(DATA_DIR, 'valid.csv')
TEST_DF_PATH = os.path.join(DATA_DIR, 'test.csv')

TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train_images')
VAL_IMG_DIR = os.path.join(DATA_DIR, 'val_images')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test_images')

IMG_SIZE = 224
BATCH_SIZE = 32

print("Libraries successfully imported and paths configured!")

## 📊 Step 3: Load CSVs & Class Weights
Because our dataset is heavily imbalanced, compute class weights so that rarer stages (like Severe) get more weight during training.

In [ ]:
# Load labels
train_df = pd.read_csv(TRAIN_DF_PATH)
val_df = pd.read_csv(VAL_DF_PATH)
test_df = pd.read_csv(TEST_DF_PATH)

# Add .png suffix to id_code to match file names
train_df['filename'] = train_df['id_code'].astype(str) + '.png'
val_df['filename'] = val_df['id_code'].astype(str) + '.png'
test_df['filename'] = test_df['id_code'].astype(str) + '.png'

# Convert diagnosis to string for Keras categorical mode
train_df['label'] = train_df['diagnosis'].astype(str)
val_df['label'] = val_df['diagnosis'].astype(str)
test_df['label'] = test_df['diagnosis'].astype(str)

# Compute Class Weights
classes = np.unique(train_df['diagnosis'].values)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_df['diagnosis'].values)
class_weights = dict(zip(classes, weights))

print("Class distribution in train set:")
print(train_df['diagnosis'].value_counts().sort_index())
print("\nComputed class weights:")
for cls, w in class_weights.items():
    print(f"  Class {cls}: {w:.4f}")

## 🔄 Step 4: Data Augmentation & Generators
We augment our training images on-the-fly (flips, rotations, zooms) to prevent overfitting and make our model robust. 
We use `preprocess_input` to properly normalize images for EfficientNetB0 during generator initialization.

In [ ]:
# Augmentation for training using EfficientNet's unique preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=180,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='constant',
    cval=128  # Mid-grey boundary padding
)

# Validation & Test Generators (no augmentation, only normalization via preprocess_input)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=TRAIN_IMG_DIR,
    x_col='filename',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=VAL_IMG_DIR,
    x_col='filename',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=TEST_IMG_DIR,
    x_col='filename',
    y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

## 📈 Step 5: Advanced Evaluator Callback (Quadratic Weighted Kappa)
Diabetic retinopathy classification models are evaluated using Cohen's Quadratic Weighted Kappa (QWK). Standard validation loss/accuracy don't capture this well. We build a custom callback that calculates the exact QWK metric on validation data at the end of every epoch.

In [ ]:
class CohenKappaCallback(Callback):
    """
    Computes Quadratic Weighted Kappa (QWK) on the validation set at the end of each epoch.
    Saves weights if val QWK improves.
    """
    def __init__(self, val_gen, save_path):
        super(CohenKappaCallback, self).__init__()
        self.val_gen = val_gen
        self.save_path = save_path
        self.best_kappa = -1.0

    def on_epoch_end(self, epoch, logs=None):
        self.val_gen.reset()
        y_true = self.val_gen.classes
        preds = self.model.predict(self.val_gen, verbose=0)
        y_pred = np.argmax(preds, axis=1)
        
        # Calculate Quadratic Weighted Kappa
        kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
        print(f"\n - Epoch {epoch+1} - val_cohen_kappa: {kappa:.4f}")
        
        # Save if improved
        if kappa > self.best_kappa:
            print(f"   🔥 Kappa improved from {self.best_kappa:.4f} to {kappa:.4f}! Saving weights...")
            self.best_kappa = kappa
            self.model.save(self.save_path)


## 🧠 Step 6: Create EfficientNetB0 Model
We pull EfficientNetB0 with ImageNet pre-trained weights and append our own custom classification layers on top.

In [ ]:
def build_model(num_classes=5):
    # Import the base pre-trained model
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # Add dense custom head for our specific classes
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)        # Heavy dropout to counter overfitting
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)
    return model, base_model

model, base_model = build_model()
model.summary()

## ❄️ Step 7: STAGE A — Warm Up Classification Head (Frozen Base)
We freeze all lower layers of EfficientNetB0 so we train solely our new head. This prevents ruining our pretrained weights early on.

In [ ]:
# Freeze EfficientNetB0 base
base_model.trainable = False

# Optimizer & Loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
checkpoint_path = '/content/best_model_stageA.keras'
kappa_callback = CohenKappaCallback(val_generator, checkpoint_path)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("🚀 Starting Stage A (Warm-up dense layers for 10 epochs)...\n")

history_A = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=[kappa_callback, early_stop]
)

## 🔓 Step 8: STAGE B — Selective Fine-Tuning
We unfreeze the top blocks of EfficientNetB0 (blocks 6 and 7) while keeping bottom layers frozen to allow the high-level features to specialize for retinal structures.

In [ ]:
# Reload the best model from stage A
if os.path.exists('/content/best_model_stageA.keras'):
    model = tf.keras.models.load_model('/content/best_model_stageA.keras')
    print("Loaded best model from Stage A.")

# Unfreeze top convolutions (EfficientNetB0 has ~237 total layers)
# Unfreezing layers from index 180 onwards
base_model.trainable = True
for layer in base_model.layers[:180]:
    layer.trainable = False

# Compile with smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_path_B = '/content/best_model_stageB.keras'
kappa_callback_B = CohenKappaCallback(val_generator, checkpoint_path_B)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
early_stop_B = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

print("🚀 Starting Stage B (Unfrozen top convolutions, training 15 epochs)...\n")

history_B = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=[kappa_callback_B, reduce_lr, early_stop_B]
)

## 🔓 Step 9: STAGE C — Full Network Fine-Tuning
Unfreeze the entire network to make the base parameters adapt to specific spots of lesions or hemorrhages, using a tiny learning rate.

In [ ]:
# Reload the best model from stage B
if os.path.exists('/content/best_model_stageB.keras'):
    model = tf.keras.models.load_model('/content/best_model_stageB.keras')
    print("Loaded best model from Stage B.")

# Unfreeze everything
for layer in model.layers:
    layer.trainable = True

# Tiny learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_path_C = '/content/best_model_final.keras'
kappa_callback_C = CohenKappaCallback(val_generator, checkpoint_path_C)
early_stop_C = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

print("🚀 Starting Stage C (Full fine-tuning, training 10 epochs)...\n")

history_C = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=[kappa_callback_C, early_stop_C]
)

## 📊 Step 10: Performance Evaluation (On Test Split)
Let's check the test set predictions, draw a confusion matrix, and generate our classification report.

In [ ]:
# Load best final weights
final_model_path = '/content/best_model_final.keras'
if not os.path.exists(final_model_path):
    # Fall back to stage B if stage C didn't improve
    final_model_path = '/content/best_model_stageB.keras'

model = tf.keras.models.load_model(final_model_path)
print(f"Evaluating using: {final_model_path}")

# Evaluate on test generator
test_generator.reset()
y_true = test_generator.classes
preds = model.predict(test_generator)
y_pred = np.argmax(preds, axis=1)

# Calculate metrics
acc = np.mean(y_true == y_pred) * 100
kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')

print(f"\nTest Accuracy: {acc:.2f}%")
print(f"Test Cohen's Quadratic Kappa: {kappa:.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'],
            yticklabels=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Classification report
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']))

## 💾 Step 11: Save Model weights back to Google Drive
Once training finishes, copy the best weights file back to drive so we can download it to our local project structure.

In [ ]:
import os
import shutil

drive_output_dir = '/content/drive/MyDrive/NammaRetina/models/'
os.makedirs(drive_output_dir, exist_ok=True)

# Identify best weights file
best_local_file = '/content/best_model_final.keras'
if not os.path.exists(best_local_file):
    best_local_file = '/content/best_model_stageB.keras'

copy_destination = os.path.join(drive_output_dir, 'efficientnetb0_dr.keras')
shutil.copy(best_local_file, copy_destination)

print(f"✅ Successfully backed up best weights to Google Drive!")
print(f"Location: {copy_destination}")
print("🎉 You can now close Colab and download the file locally to NammaRetina/models/")